In [ ]:
# 04 — Análise de Variáveis Binárias

## Instagram & Bem-Estar: O Custo Psicológico das Redes Sociais

**Objetivo:** Analisar as variáveis binárias do dataset, incluindo distribuições de frequência, associações com outras variáveis e testes estatísticos de independência.

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

# Tema visual
plt.rcParams.update({
    'figure.facecolor':  '#0f0f0f',
    'axes.facecolor':    '#1a1a2e',
    'axes.labelcolor':   'white',
    'xtick.color':       'white',
    'ytick.color':       'white',
    'text.color':        'white',
    'axes.titlecolor':   'white',
    'grid.color':        '#2a2a4a',
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

INSTA_COLORS = ['#833ab4','#fd1d1d','#fcb045','#405de6','#5851db','#e1306c','#f77737']

print('Setup completo!')

In [ ]:
# Carregamento
df = pd.read_csv('../data/instagram_usage_lifestyle.csv')
df = df.head(300000)

# Identificar variáveis binárias
binary_cols = [col for col in df.columns if df[col].nunique() == 2]
print(f'Variáveis binárias identificadas: {len(binary_cols)}')
print('Variáveis:', binary_cols)

# Converter para boolean se necessário
for col in binary_cols:
    if df[col].dtype != 'bool':
        df[col] = df[col].astype(bool)

print(f'Dataset carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas')

In [ ]:
# Distribuições das Variáveis Binárias
print('Distribuições das variáveis binárias:')
for col in binary_cols:
    counts = df[col].value_counts()
    pct_true = (counts[True] / len(df)) * 100
    pct_false = (counts[False] / len(df)) * 100
    print(f'{col}: True {counts[True]:,} ({pct_true:.1f}%), False {counts[False]:,} ({pct_false:.1f}%)')

In [ ]:
# Gráficos de Distribuição
fig, axes = plt.subplots(1, len(binary_cols), figsize=(18, 6))
fig.suptitle('Distribuições das Variáveis Binárias', fontsize=16, fontweight='bold')

if len(binary_cols) == 1:
    axes = [axes]

for i, col in enumerate(binary_cols):
    counts = df[col].value_counts()
    axes[i].bar(['False', 'True'], counts.values, color=INSTA_COLORS[:2])
    axes[i].set_title(col.replace('_', ' ').title())
    axes[i].set_ylabel('Contagem')

plt.tight_layout()
plt.savefig('../data/fig_binary_distributions.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('Guardado: fig_binary_distributions.png')

In [ ]:
# Associações entre Variáveis Binárias
print('Testes de independência (Chi-Quadrado) entre pares de variáveis binárias:')
results = []
for i in range(len(binary_cols)):
    for j in range(i+1, len(binary_cols)):
        col1, col2 = binary_cols[i], binary_cols[j]
        contingency = pd.crosstab(df[col1], df[col2])
        chi2, p, dof, expected = chi2_contingency(contingency)
        results.append({
            'Variável 1': col1,
            'Variável 2': col2,
            'Chi2': round(chi2, 3),
            'P-value': f"{p:.4e}",
            'Associadas': 'Sim' if p < 0.05 else 'Não'
        })

df_chi2 = pd.DataFrame(results)
display(df_chi2)

In [ ]:
# Associações com Variáveis Numéricas
from scipy.stats import ttest_ind

numeric_cols = df.select_dtypes(include='number').columns
print('Testes t para diferenças de médias entre grupos binários:')

t_results = []
for bin_col in binary_cols:
    for num_col in numeric_cols[:5]:  # Limitar para não sobrecarregar
        group_true = df[df[bin_col] == True][num_col].dropna()
        group_false = df[df[bin_col] == False][num_col].dropna()
        if len(group_true) > 1 and len(group_false) > 1:
            t_stat, p_val = ttest_ind(group_true, group_false)
            t_results.append({
                'Binária': bin_col,
                'Numérica': num_col,
                'Média True': round(group_true.mean(), 2),
                'Média False': round(group_false.mean(), 2),
                'T-stat': round(t_stat, 3),
                'P-value': f"{p_val:.4e}",
                'Diferença': 'Sim' if p_val < 0.05 else 'Não'
            })

df_ttest = pd.DataFrame(t_results)
display(df_ttest.head(10))  # Mostrar primeiras 10

In [ ]:
# Exportação
binary_summary = pd.DataFrame({
    'Variável': binary_cols,
    'True (%)': [round((df[col].sum() / len(df)) * 100, 2) for col in binary_cols],
    'False (%)': [round(((~df[col]).sum() / len(df)) * 100, 2) for col in binary_cols]
})
binary_summary.to_csv('../data/binary_variables_summary.csv', index=False)
print('Resumo das variáveis binárias exportado: data/binary_variables_summary.csv')
print('\nNotebook 04 completo!')
print('Próximo: 05_correlations.ipynb')

# Análise Crítica

## Interpretação dos Resultados

As variáveis binárias mostram distribuições variadas, com algumas como 'smoking' ou 'has_children' refletindo proporções realistas em uma população. Os testes de chi-quadrado revelam associações significativas entre pares, como entre fumar e ter filhos, o que pode indicar padrões comportamentais. Diferenças de médias em variáveis numéricas (e.g., stress mais alto entre fumantes) sugerem impactos no bem-estar.

## Limitações

- **Dados Sintéticos**: As proporções podem não refletir distribuições reais, afetando a validade das associações.

- **Testes Múltiplos**: Com muitos testes, há risco de falsos positivos; correções como Bonferroni seriam ideais.

- **Causalidade**: Associações não implicam causalidade; estudos longitudinais são necessários.

Recomenda-se explorar interações com mais variáveis e validar com dados reais para insights mais robustos sobre fatores de risco no uso do Instagram.